In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Generate or define a,b and x1,x2
# np.random.seed(0)  # reproducible

# Random scalars
a = 5*np.random.rand(1)[0]
a = 1.0
b = 5*np.random.rand(1)[0]

# Random unit vectors in R^2
theta_a = 0  # fixed angle in degrees
teta_a = np.deg2rad(theta_a)
v1 = np.array([np.cos(teta_a), np.sin(teta_a)])
v1 /= np.linalg.norm(v1)

theta = 90  # random angle in degrees
teta = np.deg2rad(theta)
v2 = np.array([np.cos(teta), np.sin(teta)])
v2 /= np.linalg.norm(v2)

print("a =", a, "b =", b)
print("v1 =", v1, "v2 =", v2)

# Step 2: Build the matrix
A = a * np.outer(v1, v1) + b * np.outer(v2, v2)
print("Matrix A:\n", A)

V = np.column_stack((v1, v2))
r = np.linalg.solve(V.T, np.array([a, b]))
# r /= np.linalg.norm(r)

# Step 3: Eigen decomposition
eigvals, eigvecs = np.linalg.eigh(A)  # eigvecs columns are eigenvectors
print("Eigenvalues =", eigvals)
print("Eigenvectors:\n", eigvecs)

# Step 4: Plot
plt.figure(figsize=(6,6))
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)

origin = np.array([0,0])

# -- Show reconstructed contribution vectors ---
plt.quiver(*origin, *r / np.linalg.norm(r), angles='xy', scale_units='xy', scale=1, color='purple',
           width=0.01, label=f"r1·x1 (r1={r[0]:.2f})")

# --- Show original contribution vectors ---
plt.quiver(*origin, *(a*v1), angles='xy', scale_units='xy', scale=1, color='blue',
           width=0.01, label=f"a·x1 (a={a:.2f})")
plt.quiver(*origin, *(b*v2), angles='xy', scale_units='xy', scale=1, color='green',
           width=0.01, label=f"b·x2 (b={b:.2f})")

# --- Show eigenvectors scaled by eigenvalues ---
colors = ['red','orange']
for val, vec, col in zip(eigvals, eigvecs.T, colors):
    # if col == 'red':
    #     continue
    # if vec[0] < 0:  # flip to first quadrant for better visualization
    #     vec = -vec
    plt.quiver(*origin, *(vec), angles='xy', scale_units='xy', scale=1,
               color=col, width=0.01, label=f"λ={val:.2f}")


def solve_params(A):
    assert A.shape == (2,2), "Matrix A must be 2x2"
    assert np.allclose(A, A.T), "Matrix A must be symmetric"
    C11, C12 = A[0,0], A[0,1]
    C22 = A[1,1]
    """
    Solve for a, b, theta from:
        C11 = a + b cos^2(theta)
        C12 = b cos(theta) sin(theta)
        C22 = b sin^2(theta)

    Returns:
        a, b, theta  (theta in radians)
    """

    # Handle special cases
    if np.isclose(C22, 0) and np.isclose(C12, 0):
        # Then sin(theta)=0 => theta=0
        theta = 0.0
        b = 0.0
        a = C11
        return a, b, theta

    if np.isclose(C12, 0):
        # Then cos(theta)=0 => theta=pi/2
        theta = np.pi / 2
        b = C22
        a = C11
        return a, b, theta

    # General case
    theta = np.arctan2(C22, C12)   # safer than arctan(C22/C12)
    s, c = np.sin(theta), np.cos(theta)

    # Compute b and a
    b = C22 / (s**2) if not np.isclose(s, 0) else C12 / (c*s)
    a = C11 - b * (c**2)

    return a, b, np.array([np.cos(theta), np.sin(theta)])

vec_1 = np.array([1,0])
a, b, vec_2 = solve_params(A)

# --- Show original contribution vectors ---
# plt.quiver(*origin, *(a*vec_1), angles='xy', scale_units='xy', scale=1, color='purple',
#            width=0.01, label=f"a·v1 (a={a:.2f})")
# plt.quiver(*origin, *(b*vec_2), angles='xy', scale_units='xy', scale=1, color='yellow',
#            width=0.01, label=f"b·v2 (b={b:.2f})")

print("a =", a)
print("b =", b)
print("theta (rad) =", theta)
print("theta (deg) =", np.degrees(theta))


# Formatting
plt.xlim(-5,5)
plt.ylim(-5,5)
plt.gca().set_aspect('equal', adjustable='box')
plt.legend()
plt.title("Original scaled vectors (a·x1, b·x2) vs Eigen-decomposition (λ·v)")
plt.show()
